## Step1:Importing modules

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import SparkSession
from pyspark.sql.window import Window

**Insight**

Importing the required PySpark modules for data processing. These modules provide functions for transformations, data types, creating a Spark session, and applying window functions for  operations like ranking.

## Step2:Create Spark Session

In [0]:
sess = SparkSession.builder \
        .appName("celebal_assignment") \
        .getOrCreate()

**Insight**

Creating spark session with name "celebal_assignment"

## Step3:Load the dataset

In [0]:
data=sess.read.format("csv")\
    .option("header","true")\
        .option("inferSchema","true")\
            .load("/Volumes/dbacademy/default/data/employee_data.csv")



**Insight**

Loading the employee dataset  a CSV file into a Spark DataFrame. The header option uses the first row as column names, while inferSchema automatically detects the suitable data type for each column.

## Step4:Basic details

In [0]:
data.printSchema()     

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- department: string (nullable = true)
 |-- region: string (nullable = true)
 |-- category: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- bonus: integer (nullable = true)
 |-- sales: integer (nullable = true)
 |-- experience: integer (nullable = true)
 |-- joining_date: date (nullable = true)



In [0]:
data.show()

+------+------+----+------+----------+------+--------+------+-----+------+----------+------------+
|emp_id|  name| age|gender|department|region|category|salary|bonus| sales|experience|joining_date|
+------+------+----+------+----------+------+--------+------+-----+------+----------+------------+
|   101|  Amit|  25|  Male|     Sales| North|       A| 45000| 5000|120000|         2|  2023-01-15|
|   102| Priya|  29|Female|        HR|  West|       B| 55000| 6000|     0|         5|  2020-03-12|
|   103| Rahul|  31|  Male|        IT| South|       A| 65000| 8000| 95000|         7|  2018-07-22|
|   104| Sneha|NULL|Female|   Finance|  East|       C| 70000| 7000| 87000|         6|  2019-08-14|
|   105| Karan|  27|  Male|     Sales| North|       B| 48000| NULL|110000|         3|  2022-05-10|
|   106|  Neha|  26|Female| Marketing|  West|       A| 52000| 4000| 65000|         2|  2023-02-18|
|   107| Vikas|  45|  Male|        IT| South|       C| 90000|10000|150000|        15|  2010-11-01|
|   108|  

In [0]:
data.columns

['emp_id',
 'name',
 'age',
 'gender',
 'department',
 'region',
 'category',
 'salary',
 'bonus',
 'sales',
 'experience',
 'joining_date']

In [0]:
print(f"Rows are :{data.count()}")
print(f"Columns are :{len(data.columns)}")

Rows are :30
Columns are :12


**Insight**

Showing the basic information of the dataset like the schema, column names, and dimensions, to understand its structure before performing data cleaning and analysis.

## Step5:Dataframe Immutable Prove

In [0]:
data1=data.withColumn("Country",lit("india"))

In [0]:
data.show(5)

+------+-----+----+------+----------+------+--------+------+-----+------+----------+------------+
|emp_id| name| age|gender|department|region|category|salary|bonus| sales|experience|joining_date|
+------+-----+----+------+----------+------+--------+------+-----+------+----------+------------+
|   101| Amit|  25|  Male|     Sales| North|       A| 45000| 5000|120000|         2|  2023-01-15|
|   102|Priya|  29|Female|        HR|  West|       B| 55000| 6000|     0|         5|  2020-03-12|
|   103|Rahul|  31|  Male|        IT| South|       A| 65000| 8000| 95000|         7|  2018-07-22|
|   104|Sneha|NULL|Female|   Finance|  East|       C| 70000| 7000| 87000|         6|  2019-08-14|
|   105|Karan|  27|  Male|     Sales| North|       B| 48000| NULL|110000|         3|  2022-05-10|
+------+-----+----+------+----------+------+--------+------+-----+------+----------+------------+
only showing top 5 rows


In [0]:
data1.show(5)

+------+-----+----+------+----------+------+--------+------+-----+------+----------+------------+-------+
|emp_id| name| age|gender|department|region|category|salary|bonus| sales|experience|joining_date|Country|
+------+-----+----+------+----------+------+--------+------+-----+------+----------+------------+-------+
|   101| Amit|  25|  Male|     Sales| North|       A| 45000| 5000|120000|         2|  2023-01-15|  india|
|   102|Priya|  29|Female|        HR|  West|       B| 55000| 6000|     0|         5|  2020-03-12|  india|
|   103|Rahul|  31|  Male|        IT| South|       A| 65000| 8000| 95000|         7|  2018-07-22|  india|
|   104|Sneha|NULL|Female|   Finance|  East|       C| 70000| 7000| 87000|         6|  2019-08-14|  india|
|   105|Karan|  27|  Male|     Sales| North|       B| 48000| NULL|110000|         3|  2022-05-10|  india|
+------+-----+----+------+----------+------+--------+------+-----+------+----------+------------+-------+
only showing top 5 rows


**Insight**

Proving the immutable nature of Spark DataFrames by showing that transformation operations create a new DataFrame instead of modifying the original.

## Step6:Data Cleaning

In [0]:
data.count()

30

In [0]:
data=data.drop_duplicates(["emp_id"])

In [0]:
data.count()

28

**Insight**

Checked the total number of records before and after applying drop_duplicates()  to identify and remove duplicate employee entries.

## Handle Null Values

In [0]:
data.select(
    [count(when(col(i).isNull(), i)).alias(i) for i in data.columns]
).show()

+------+----+---+------+----------+------+--------+------+-----+-----+----------+------------+
|emp_id|name|age|gender|department|region|category|salary|bonus|sales|experience|joining_date|
+------+----+---+------+----------+------+--------+------+-----+-----+----------+------------+
|     0|   1|  2|     0|         0|     2|       0|     1|    2|    1|         0|           0|
+------+----+---+------+----------+------+--------+------+-----+-----+----------+------------+



In [0]:
data_clean=data.dropna()

In [0]:
data=data.fillna({
    "name":"unknown",
    "age":0,
    "region":"unknown",
    "gender":"unknown",
    "department":"unknown",
    "category":"unknown",
    "salary":0,
    "bonus":0,
    "sales":0,
    "experience":0,
    "joining_date":"0000-00-00"
})

In [0]:
data=data.withColumn("name",when(col("name")=="","unknown").otherwise(col("name")))
data=data.withColumn("region",when(col("region")=="","unknown").otherwise(col("region")))
data=data.withColumn("gender",when(col("gender")=="","unknown").otherwise(col("gender")))
data=data.withColumn("department",when(col("department")=="","unknown").otherwise(col("department")))
data=data.withColumn("category",when(col("category")=="","unknown").otherwise(col("category")))


## Filtering

In [0]:
data = data.filter(col("emp_id") > 0)
data=data.filter(col("name")!="")
data = data.filter(col("salary") > 0)

**Insight**

Handled missing values using dropna() to remove incomplete records and fillna() to replace null values with appropriate values.And apply filter
to filter out the unwanted values and data. 

## Handle Data Types

In [0]:
data.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = false)
 |-- age: integer (nullable = false)
 |-- gender: string (nullable = false)
 |-- department: string (nullable = false)
 |-- region: string (nullable = false)
 |-- category: string (nullable = false)
 |-- salary: integer (nullable = false)
 |-- bonus: integer (nullable = false)
 |-- sales: integer (nullable = false)
 |-- experience: integer (nullable = false)
 |-- joining_date: date (nullable = true)



In [0]:
data=data.withColumn("sales",col("sales").cast("integer"))
data=data.withColumn("experience",col("experience").cast("integer"))

**Insight**

Converted columns to appropriate data types to ensure proper calculations, filtering, and aggregations.

## Step7:Aggregate Function Operation

In [0]:
print("Total employees ")
data.count()

Total employees 


27

In [0]:
print("Total Bonus given to Employees")
data.select(sum("bonus").alias("total_bonus")).show()

Total Bonus given to Employees
+-----------+
|total_bonus|
+-----------+
|     156000|
+-----------+



In [0]:
print("Total Salary given to Employees")
data.select(sum("salary").alias("total_bonus")).show()

Total Salary given to Employees
+-----------+
|total_bonus|
+-----------+
|    1718000|
+-----------+



In [0]:
print("Maximum salary employee")
data.filter(col("salary") == data.select(max("salary")).collect()[0][0]).show()

Maximum salary employee
+------+-----+---+------+----------+------+--------+------+-----+------+----------+------------+
|emp_id| name|age|gender|department|region|category|salary|bonus| sales|experience|joining_date|
+------+-----+---+------+----------+------+--------+------+-----+------+----------+------------+
|   107|Vikas| 45|  Male|        IT| South|       C| 90000|10000|150000|        15|  2010-11-01|
+------+-----+---+------+----------+------+--------+------+-----+------+----------+------------+



In [0]:
print("Minimum salary employee")
data.filter(col("salary") == data.select(min("salary")).collect()[0][0]).show()

Minimum salary employee
+------+-----+---+------+----------+------+--------+------+-----+-----+----------+------------+
|emp_id| name|age|gender|department|region|category|salary|bonus|sales|experience|joining_date|
+------+-----+---+------+----------+------+--------+------+-----+-----+----------+------------+
|   116|Nisha| 22|Female| Marketing|  West|       A| 43000| 2500|42000|         1|  2024-02-10|
+------+-----+---+------+----------+------+--------+------+-----+-----+----------+------------+



**Insight**

Applied aggregate functions such as count(),sum(), max(), and min(). These operations help to calculate total values and identify the highest and lowest values for better analysis.

## Step8:Data Grouping and Aggregation

In [0]:
data.groupBy("department").agg(sum("salary").alias("total_salary")).show()

+----------+------------+
|department|total_salary|
+----------+------------+
|     Sales|      400000|
|   Finance|      437000|
|        HR|      226000|
|        IT|      362000|
| Marketing|      293000|
+----------+------------+



In [0]:
data.groupBy("department").agg(sum("bonus").alias("total_bonus")).show()
data.groupBy("department").agg(sum("sales").alias("total_sales")).show()

+----------+-----------+
|department|total_bonus|
+----------+-----------+
|     Sales|      35500|
|   Finance|      39500|
|        HR|      17000|
|        IT|      38500|
| Marketing|      25500|
+----------+-----------+

+----------+-----------+
|department|total_sales|
+----------+-----------+
|     Sales|     861000|
|   Finance|     386000|
|        HR|      55000|
|        IT|     570000|
| Marketing|     373000|
+----------+-----------+



In [0]:
print("Department with maximum salary ")
data.groupBy("department").agg(sum("salary").alias("total_salary"))\
    .filter(col("total_salary")==data.groupBy("department").agg(sum("salary").alias("total_salary")).select(max("total_salary")).collect()[0][0]).show()

Department with maximum salary 
+----------+------------+
|department|total_salary|
+----------+------------+
|   Finance|      437000|
+----------+------------+



In [0]:
print("Department with minimum sales")
data.groupBy("department").agg(sum("sales").alias("total_sales"))\
    .filter(col("total_sales")==data.groupBy("department").agg(sum("sales").alias("total_sales")).select(max("total_sales")).collect()[0][0]).show()

Department with minimum sales
+----------+-----------+
|department|total_sales|
+----------+-----------+
|     Sales|     861000|
+----------+-----------+



**Insight**

Grouped employee records by department and calculated the total salary, bonus, and sales using aggregate functions.

## Transformations

### Wide transformation 

In [0]:
data.groupBy("department").agg(sum("salary").alias("total_salary")).show()

+----------+------------+
|department|total_salary|
+----------+------------+
|     Sales|      400000|
|   Finance|      437000|
|        HR|      226000|
|        IT|      362000|
| Marketing|      293000|
+----------+------------+



In [0]:
data.orderBy(col("salary").desc()).show(6)

+------+------+---+------+----------+------+--------+------+-----+------+----------+------------+
|emp_id|  name|age|gender|department|region|category|salary|bonus| sales|experience|joining_date|
+------+------+---+------+----------+------+--------+------+-----+------+----------+------------+
|   107| Vikas| 45|  Male|        IT| South|       C| 90000|10000|150000|        15|  2010-11-01|
|   115|Deepak| 41|  Male|     Sales| North|       B| 85000| 9000|180000|        14|  2011-04-04|
|   127| Akash| 40|  Male| Marketing| South|       C| 83000| 9000|135000|        13|  2012-08-15|
|   123|Rakesh| 39|  Male|   Finance|  East|       C| 81000| 8500| 90000|        12|  2013-05-18|
|   117|  Ajay| 36|  Male|        IT| South|       C| 78000| 8000|125000|        11|  2014-07-13|
|   121|Manish| 35|  Male|     Sales| North|       B| 76000| 8500|140000|         9|  2016-06-16|
+------+------+---+------+----------+------+--------+------+-----+------+----------+------------+
only showing top 6 r

**Insight**

Performing wide transformations using groupBy() and orderBy().These operations involve data shuffling across partitions to perform grouping and sorting

### Narrow Tranformation

In [0]:
data.select("name").show(3)

+------+
|  name|
+------+
| Karan|
|Manish|
| Sneha|
+------+
only showing top 3 rows


In [0]:
data.where(col("salary")>80000).show()

+------+------+---+------+----------+------+--------+------+-----+------+----------+------------+
|emp_id|  name|age|gender|department|region|category|salary|bonus| sales|experience|joining_date|
+------+------+---+------+----------+------+--------+------+-----+------+----------+------------+
|   123|Rakesh| 39|  Male|   Finance|  East|       C| 81000| 8500| 90000|        12|  2013-05-18|
|   115|Deepak| 41|  Male|     Sales| North|       B| 85000| 9000|180000|        14|  2011-04-04|
|   107| Vikas| 45|  Male|        IT| South|       C| 90000|10000|150000|        15|  2010-11-01|
|   127| Akash| 40|  Male| Marketing| South|       C| 83000| 9000|135000|        13|  2012-08-15|
+------+------+---+------+----------+------+--------+------+-----+------+----------+------------+



**Insight**

Performed narrow transformations using select() and where() to retrieve specific columns and filter records

## Step10:Complete Pipeline

In [0]:
complete_pipeline = (
    data.dropDuplicates()
        
        .fillna({
            "name": "unknown",
            "age": 0,
            "region": "unknown",
            "gender": "unknown",
            "department": "unknown",
            "category": "unknown",
           
        })
        .filter(col("emp_id") > 0)
        .filter(col("name") != "")
        .filter(col("salary") > 0)
        .withColumn("sales", col("sales").cast("integer"))
        .withColumn("experience", col("experience").cast("integer"))
        .groupBy("department").agg(
    sum("salary").alias("total_salary"),
    sum("bonus").alias("total_bonus"),
    sum("sales").alias("total_sales")
)
)

complete_pipeline.show()

+----------+------------+-----------+-----------+
|department|total_salary|total_bonus|total_sales|
+----------+------------+-----------+-----------+
|     Sales|      400000|      35500|     861000|
|   Finance|      437000|      39500|     386000|
|        HR|      226000|      17000|      55000|
|        IT|      362000|      38500|     570000|
| Marketing|      293000|      25500|     373000|
+----------+------------+-----------+-----------+



**Insight**

The complete pipeline combines data cleaning, filtering, data type conversion. First, duplicate records and missing values are handled to improve data quality. Then, invalid records are filtered, and the required columns are converted to the correct data types. Finally, the data is grouped by department to calculate the total salary, total bonus, and total sales, providing a clear summary for each department.